In [2]:
import pandas as pd
import numpy as np

np.random.seed(42)
n_samples = 3000

# Generate raw features
data = {
    'Temperature': np.random.normal(25, 8, n_samples),
    'Humidity': np.random.uniform(10, 95, n_samples),
    'WindSpeed': np.random.exponential(5, n_samples),
    'HistoricalLoad': np.random.gamma(shape=2, scale=500, size=n_samples),
    'OperationalShift': np.random.choice(['Morning', 'Evening', 'Night', 'Maintenance'], n_samples, p=[0.4, 0.3, 0.2, 0.1]),
    'EquipmentAge': np.random.randint(1, 15, n_samples)
}

df = pd.DataFrame(data)

# Create a non-linear target condition with noise
load_risk = (df['HistoricalLoad'] * 0.001) + (df['Temperature'] * 0.3) - (df['Humidity'] * 0.05) + (df['OperationalShift'] == 'Evening') * 3
df['CriticalOverload'] = (load_risk > np.percentile(load_risk, 75)).astype(int) # Top 25% are overloads

# Inject real-world messiness (15% missing values in numeric fields) using a clean column-wise mask
cols_to_miss = ['Temperature', 'WindSpeed', 'HistoricalLoad']
for col in cols_to_miss:
    mask = np.random.rand(n_samples) < 0.15
    df.loc[mask, col] = np.nan

# Inject random typos in categorical strings
df.loc[np.random.choice(n_samples, 20, replace=False), 'OperationalShift'] = 'Mornng' # Typo

print(f"Dataset Shape: {df.shape}")
print(df.isnull().sum())

Dataset Shape: (3000, 7)
Temperature         417
Humidity              0
WindSpeed           466
HistoricalLoad      461
OperationalShift      0
EquipmentAge          0
CriticalOverload      0
dtype: int64


In [3]:
print(df.head())

   Temperature   Humidity  ...  EquipmentAge  CriticalOverload
0          NaN  79.592920  ...            11                 0
1    23.893886  93.603213  ...            14                 0
2          NaN  32.072021  ...             9                 0
3    37.184239  92.426509  ...             5                 1
4    23.126773  46.654879  ...             4                 0

[5 rows x 7 columns]


In [4]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["CriticalOverload"])
y = df["CriticalOverload"]
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)
print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of y_train: {y_train.shape}")

Shape of X_train: (2250, 6)
Shape of y_train: (2250,)


In [5]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

numeric_features = ['Temperature', 'Humidity', 'WindSpeed', 'HistoricalLoad', 'EquipmentAge']
categorical_features = ['OperationalShift']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scalar', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('OneHot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

X_train_processsed = preprocessor.fit_transform(X_train)

In [6]:
import xgboost as xgb

dtrain = xgb.DMatrix(data = X_train_processsed, label = y_train)

params = {
    'objective' : 'binary:logistic',
    'eval_metric' : 'logloss',
    'learning_rate' : 0.1,
    'max_depth' : 6,
    'subsample' : .8 
}
cv_results = xgb.cv(
    params,
    dtrain,
    num_boost_round = 100,
    nfold = 5,
    early_stopping_rounds = 10,
    stratified = True,
    seed = 42
)

print(f"{cv_results.head()}")

   train-logloss-mean  train-logloss-std  test-logloss-mean  test-logloss-std
0            0.490103           0.001325           0.495041          0.002725
1            0.440549           0.001801           0.449466          0.003751
2            0.400201           0.001638           0.413031          0.003918
3            0.366278           0.001755           0.382140          0.004577
4            0.337230           0.001771           0.356518          0.004748


In [7]:
current_score = cv_results['test-logloss-mean'].min()
print(f"The lowest logloss for this cv = {current_score}")

The lowest logloss for this cv = 0.16173343340855922


In [8]:
dtrain = xgb.DMatrix(data = X_train_processsed, label = y_train)
best_score = 100
best_depth = 0
best_rounds = 0

depths = [2,3,4,5,6,7,8,9]

for depth in depths:
    params = {
    'objective' : 'binary:logistic',
    'eval_metric' : 'logloss',
    'learning_rate' : 0.1,
    'max_depth' : depth,
    'subsample' : .8 
    }
    cv_results = xgb.cv(
        params,
        dtrain,
        num_boost_round = 100,
        nfold = 5,
        early_stopping_rounds = 10,
        stratified = True,
        seed = 42
    )

    current_score = cv_results['test-logloss-mean'].min()
    print(f"Depth: {depth} | Best Rounds: {len(cv_results)} | LogLoss: {current_score:.5f}")

    if current_score < best_score:
        best_score = current_score
        best_depth = depth
        best_rounds = len(cv_results)

print(f"The best score = {best_score}")
print(f"The best depth = {best_depth}")
print(f"The best rounds = {best_rounds}")

Depth: 2 | Best Rounds: 100 | LogLoss: 0.18542
Depth: 3 | Best Rounds: 100 | LogLoss: 0.17052
Depth: 4 | Best Rounds: 74 | LogLoss: 0.16570
Depth: 5 | Best Rounds: 74 | LogLoss: 0.16193
Depth: 6 | Best Rounds: 71 | LogLoss: 0.16173
Depth: 7 | Best Rounds: 64 | LogLoss: 0.16406
Depth: 8 | Best Rounds: 62 | LogLoss: 0.16300
Depth: 9 | Best Rounds: 65 | LogLoss: 0.16260
The best score = 0.16173343340855922
The best depth = 6
The best rounds = 71


In [9]:
dtrain = xgb.DMatrix(data = X_train_processsed, label = y_train)
best_score = 100
best_reg_lambda = 0
best_rounds = 0

reg_lambdas = [1,3,5]

for lambda_ in reg_lambdas:
    params = {
    'objective' : 'binary:logistic',
    'eval_metric' : 'logloss',
    'learning_rate' : 0.1,
    'max_depth' : best_depth,
    'subsample' : .8,
    'reg_lambda' : lambda_
    }
    cv_results = xgb.cv(
        params,
        dtrain,
        num_boost_round = 100,
        nfold = 5,
        early_stopping_rounds = 10,
        stratified = True,
        seed = 42
    )

    current_score = cv_results['test-logloss-mean'].min()
    print(f"lambda: {lambda_} | Best Rounds: {len(cv_results)} | LogLoss: {current_score:.5f}")

    if current_score < best_score:
        best_score = current_score
        best_reg_lambda = lambda_
        best_rounds = len(cv_results)

print(f"The best score = {best_score}")
print(f"The best reg_lambda = {best_reg_lambda}")
print(f"The best rounds = {best_rounds}")

lambda: 1 | Best Rounds: 71 | LogLoss: 0.16173
lambda: 3 | Best Rounds: 77 | LogLoss: 0.16440
lambda: 5 | Best Rounds: 77 | LogLoss: 0.16645
The best score = 0.16173343340855922
The best reg_lambda = 1
The best rounds = 71


In [10]:
params = {
    'objective' : 'binary:logistic',
    'eval_metric' : 'logloss',
    'learning_rate' : 0.03,
    'max_depth' : best_depth,
    'subsample' : .8,
    'reg_lambda' : best_reg_lambda
    }
cv_results = xgb.cv(
    params,
    dtrain,
    num_boost_round = 1500,
    nfold = 5,
    early_stopping_rounds = 30,
    stratified = True,
    seed = 42
)


current_score = cv_results['test-logloss-mean'].min()
print(f"Best Rounds: {len(cv_results)} | LogLoss: {current_score:.5f}")

learning_rate = .03
best_rounds = len(cv_results)

Best Rounds: 241 | LogLoss: 0.16127


In [11]:
X_train_processsed = preprocessor.transform(X_train)

dtrain = xgb.DMatrix(data=X_train_processsed, label=y_train)

final_params = {
    'objective' : 'binary:logistic',
    'eval_metric' : 'logloss',
    'learning_rate' : 0.03,
    'max_depth' : 6,
    'subsample' : .8,
    'reg_lambda' : 1
}

final_model = xgb.train(
    params = final_params,
    dtrain = dtrain,
    num_boost_round = 241
)

In [14]:
from sklearn.metrics import log_loss, roc_auc_score, f1_score, confusion_matrix

X_test_processed = preprocessor.transform(X_test)
dtest = xgb.DMatrix(data = X_test_processed, label=y_test)

y_prob = final_model.predict(dtest)
y_pred = y_prob > .5

print(f"Final Test LogLoss: {log_loss(y_test, y_prob):.5f}")
print(f"Final Test ROC-AUC: {roc_auc_score(y_test, y_prob):.5f}")
print(f"Final f1 score = {f1_score(y_test, y_pred)}")
print(f"Final confustion matrix = {confusion_matrix(y_test, y_pred)}")

Final Test LogLoss: 0.13348
Final Test ROC-AUC: 0.98617
Final f1 score = 0.9095477386934674
Final confustion matrix = [[533  12]
 [ 24 181]]
